# Notebook 04 — Capstone: multi-agent deep research

**Workshop:** Agentic AI for Scientists — Week 3 (From Loops to Graphs)
**Block:** Capstone + Week 4 bridge (15 min) — *driven, not typed along*
**Goal:** Build a **minimal** supervisor → {researcher, writer} graph that does an iterative literature review, then map its shape onto the real systems you'll meet in the wild: `open_deep_research`, GPT-Researcher, and **TriAgent**. The lesson: multi-agent is **specialisation**, not headcount — you reach for it when one agent's context or skillset isn't enough.


## 0. Setup

In [ ]:
%pip install -q \
    "langgraph>=1.2.4" "langchain>=1.3.2" "langchain-core>=1.4.0" \
    "langchain-google-genai>=4.2.4" python-dotenv

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# Accept GEMINI_API_KEY as an alias (some AI Studio users store it under that name).
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


## 1. The shape: a supervisor that delegates

One **supervisor** decomposes a research question into sub-topics. A **researcher** worker answers each sub-topic (here from the model's knowledge + an optional search; in a real system, from a retriever or the web). A **writer** worker synthesises the findings into a report. The supervisor decides when there's enough to write.

State carries the question, the list of open sub-topics, the accumulated findings, and the final report.


In [ ]:
from typing import TypedDict, List
import operator
from typing import Annotated
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


class ResearchState(TypedDict):
    question: str
    subtopics: List[str]
    findings: Annotated[list, operator.add]
    report: str


class Plan(BaseModel):
    subtopics: list[str] = Field(description="3 focused sub-questions that together answer the main question.")


planner = llm.with_structured_output(Plan)


## 2. The nodes — supervisor, researcher, writer

In [ ]:
def supervisor(state: ResearchState) -> dict:
    # plan once, at the start
    if not state["subtopics"]:
        plan = planner.invoke(f"Break this research question into sub-questions: {state['question']}")
        return {"subtopics": plan.subtopics}
    return {}


def researcher(state: ResearchState) -> dict:
    # answer the NEXT open sub-topic (sequential here; could fan out in parallel)
    topic = state["subtopics"][len(state["findings"])]
    ans = llm.invoke(f"Answer concisely, like a literature note (3-4 sentences): {topic}").content
    return {"findings": [f"### {topic}\n{ans}"]}


def writer(state: ResearchState) -> dict:
    body = "\n\n".join(state["findings"])
    report = llm.invoke(
        f"Write a short structured research brief answering '{state['question']}' "
        f"from these notes:\n\n{body}"
    ).content
    return {"report": report}


def route(state: ResearchState) -> str:
    # keep researching until every sub-topic has a finding, then write
    if len(state["findings"]) < len(state["subtopics"]):
        return "research"
    return "write"


## 3. Wire it — `supervisor → researcher ↺ → writer`

The supervisor plans; the researcher loops once per sub-topic (a cycle, as in NB00–02); the writer closes it out.


In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(ResearchState)
g.add_node("supervisor", supervisor)
g.add_node("researcher", researcher)
g.add_node("writer", writer)

g.add_edge(START, "supervisor")
g.add_edge("supervisor", "researcher")
g.add_conditional_edges("researcher", route, {"research": "researcher", "write": "writer"})
g.add_edge("writer", END)

deep_research = g.compile()

out = deep_research.invoke({
    "question": "What makes an LLM agent reliable enough for clinical decision support?",
    "subtopics": [], "findings": [], "report": "",
})
print("SUB-TOPICS:", out["subtopics"])
print("\n=== REPORT ===\n", out["report"])


**See the graph.** The supervisor → researcher → writer pipeline, with the conditional edge that loops research until the supervisor routes to the write step.

In [ ]:
# --- See the graph this code builds: Supervisor / worker deep research ---
# A compiled LangGraph app can draw its own topology: nodes are steps,
# edges are control flow. This picture IS the agent you just wired.
try:
    from IPython.display import Image, display
    display(Image(deep_research.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png needs network/graphviz; ASCII always works
    print(deep_research.get_graph().draw_ascii())

## 4. From this toy to the real systems

You just built the core pattern. Here's how the systems you'll actually use extend it:

**`langchain-ai/open_deep_research`** — its default is a *single* iterative researcher (loop search→analyze→synthesize), but it ships two reference architectures that map straight onto what you built:
- **Plan-and-Execute** = our `supervisor` (plan) + sequential `researcher`, with a **human approval** step between sections (that's NB03's `interrupt`).
- **Supervisor-Researcher** = the same graph but researchers run **in parallel** (NB03's async fan-out). Its key engineering idea: **decouple model choice from logic** via `init_chat_model()`, so each role (summarise / research / compress / write) can use a different model.

**GPT-Researcher** (assafelovic) — a *planner* generates questions, *parallel execution* agents gather per question, a *publisher* aggregates the report. Same supervisor/worker shape, STORM-style.

**TriAgent** (your instructor's framework, `krmdel/TriAgent`) — the production version of this idea for **clinical biomarker discovery**. A 4-node LangGraph pipeline:
1. **Scoping** — clarifies the goal, emits an assumptions brief.
2. **Data Analysis** — runs EDA + H2O AutoML on the cohort.
3. **Deep Research** — *exactly the supervisor→worker pattern above*: a Supervisor agent generates research topics, delegates to specialised retrieval sub-agents (Chroma + Tavily), and a Findings-Synthesis agent classifies each biomarker as **established** or **novel candidate**, with justification.
4. **Report** — synthesises a literature-grounded final report (optional PDF).

The point: **every box in TriAgent's Deep Research node is a node you built this week** — a supervisor, worker researchers, a synthesiser, conditional edges, checkpoints for human review. Multi-agent isn't a new framework to learn; it's these primitives, composed for a real problem.

---

## Recap & Week 4 bridge

- **Multi-agent = supervisor + specialised workers**, reached for when one agent's context isn't enough.
- The toy graph here is the same shape as open_deep_research, GPT-Researcher, and TriAgent — they add parallelism, role-specific models, human checkpoints, and real retrieval.
- **Deployment** (LangGraph Studio, Cloud, LangSmith Deployment) wraps a compiled graph like these in an API + UI — a slide tour today, hands-on later.

Week 4 builds on this toward [evaluation / domain agents — TBD].
